# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakeshkumarkhatri/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract

- **Unit of analysis:** One row represents one content item for one client on one reporting date.
- **Verification window:** March 1, 2026 through March 31, 2026.
- **Observed:** March 2026 contains 9,841,378 rows across 31 reporting dates.
- **Grain check:** The combination of `report_date`, `client_hash_id`, and `content_hash_id` has 9,841,378 distinct keys, matching the total row count.

In [ ]:
# Section 1 — Verify unit of analysis and time window

section1_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            CAST(report_date AS VARCHAR) || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        )) AS distinct_grain_keys,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

section1_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_keys,first_date,last_date,distinct_dates
0,9841378,9841378,2026-03-01,2026-03-31,31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


### Field contract

**Five features**

1. **`gsc_impressions`** — Feature. Knowable at the decision moment because it measures Search Console impressions from the past feature window.
2. **`gsc_clicks`** — Feature. Knowable at the decision moment because it measures Search Console clicks from the past feature window.
3. **`gsc_avg_position`** — Feature. Knowable at the decision moment because it measures average search position from the past feature window. It can be missing when GSC position data is unavailable.
4. **`ga4_sessions`** — Feature. Knowable at the decision moment because it measures GA4 sessions from the past feature window. It can be missing when GA4 data is unavailable.
5. **`ga4_engaged_sessions`** — Feature. Knowable at the decision moment because it measures engaged GA4 sessions from the past feature window. It can be missing when GA4 data is unavailable.

**Label / proxy**

- **Future content-performance outcome** — the outcome to predict after the feature window. Future performance is not used as a feature.

**Context**

- `report_date`, `month`, `client_hash_id`, and `content_hash_id` — used to define the time window, identify/group observations, and join records.
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available` — used to understand source availability and missingness.

**Excluded**

- Future-window performance measurements used to construct the label are excluded because they would not be known at the prediction moment and would cause data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Query 1 — Verify grain

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            CAST(report_date AS VARCHAR) || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        )) AS distinct_grain_keys,
        COUNT(*) - COUNT(DISTINCT (
            CAST(report_date AS VARCHAR) || '|' ||
            client_hash_id || '|' ||
            content_hash_id
        )) AS duplicate_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("Query 1 — Grain verification")
grain_check


# Query 2 — Counts, missing values and date window

data_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date) AS distinct_dates,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,

        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS gsc_impressions_nulls,

        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS gsc_clicks_nulls,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        ) AS gsc_avg_position_nulls,

        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS ga4_sessions_nulls

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("Query 2 — Counts, missing values and date window")
data_check

# Query 3 — Availability verification

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) / COUNT(*),
            2
        ) AS gsc_available_pct,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE ga4_data_available IS TRUE
            ) / COUNT(*),
            2
        ) AS ga4_available_pct

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""").df()

print("Query 3 — Availability using IS TRUE")
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — Grain verification
Query 2 — Counts, missing values and date window
Query 3 — Availability using IS TRUE


,total_rows,gsc_available_rows,ga4_available_rows,gsc_available_pct,ga4_available_pct
0,9841378,3611061,413966,36.69,4.21


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

- **Unbalanced history:** The March-to-April check found 331,436 of 331,437 March client/content pairs with an April observation. One pair has no April row, so a future-window outcome cannot be calculated for every March observation.
- **Incomplete source coverage:** March does not have complete GSC/GA4 coverage. GSC availability is TRUE for 36.69% of rows and GA4 availability is TRUE for 4.21% of rows. GA4-related fields can therefore be unavailable for a substantial part of the slice.
- **Missing values are not automatically zero:** `gsc_avg_position` is NULL for 6,230,317 March rows, while GSC impressions and clicks have no NULLs. Missingness must therefore be handled according to the field's availability rather than replaced with zero without justification.
- **Time-window limitation:** A future label requires a separate future window. Information from the future window cannot be used as a feature because it would not be available at the prediction moment and would create leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.